## Model accuracy
Import libraries

In [ ]:
import os
import json
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.constants import Avogadro as UNA

Define inputs

In [ ]:
model_name = "meno"

In [ ]:
# Models
models = {
  "meno": "./../meno/03.merged/case.r01.f64/saving/noneq/SavedModel/",
  "meno_nocorr": "./../meno/02.dist/case.r01.nocorr/saving/noneq_dist/SavedModel/",
  "deeponet_r05": "./../deeponet/02.merged/case.r05.siso/saving/noneq/SavedModel",
  "deeponet_r06": "./../deeponet/02.merged/case.r06.siso/saving/noneq/SavedModel"
}
# Paths
path_to_datagen = "./../datagen/"
paths = {
  "data": path_to_datagen + "/data/test/",
  "model": models[model_name],
  "saving": f"./output/error/{model_name}/"
}
os.makedirs(paths["saving"], exist_ok=True)
# Error
# > Epsilon value
eps = 1e-10
# > Confidence interval
alpha = 0.95
uncert_name = f"ci_{int(100*alpha)}" if (alpha > 0) else "std"

In [ ]:
# Time grid
tlim = [1e-11, 1e-1]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Define data indices

In [ ]:
# Data indices
with open(path_to_datagen + "/data/indices.json") as file:
  _indices = json.load(file)
if (model_name == "meno"):
  indices = {
    "x": [
      _indices["na_0"],
      _indices["nmi_0"],
      _indices["T_0"],
      _indices["t"],
      _indices["qoi_eq"]
    ],
    "y": [
      _indices["na"],
      _indices["nmi"],
      _indices["T"]
    ]
  }
elif model_name.startswith("deeponet"):
  indices = {
    "x": [
      _indices["na_0"],
      _indices["nmi_0"],
      _indices["T_0"],
      _indices["t"],
      _indices["qoi_eq"]
    ],
    "y": [
      _indices["na"] + _indices["nmi"],
      _indices["T"]
    ]
  }
elif (model_name == "meno_nocorr"):
  indices = {
    "x": [
      _indices["na_0"],
      _indices["nmi_0"],
      _indices["T_0"],
      _indices["t"],
      _indices["qoi_eq"],
      _indices["na"]+_indices["T"]
    ],
    "y": [
      _indices["nmi"]
    ]
  }
else:
  raise ValueError()

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])

Set variables names

In [ ]:
names = np.array(list(data.values())[0].columns)
names = np.concatenate([names[i] for i in indices["y"]]).tolist()

In [ ]:
names_latex = {}
for name in names:
  if ("O2" in name):
    names_latex[name] = "O$_2\,(%s)$" % name.split("_")[-1]
  elif ("O" in name):
    names_latex[name] = name.split("_")[-1]
  else:
    names_latex[name] = "$%s$" % name

Compute error

In [ ]:
scale = np.full(len(names), 1.0/UNA)
if (model_name != "meno_nocorr"):
  # No scaling for temperature
  scale[-1] = 1.0
scale = scale.reshape(1,-1)

In [ ]:
error = {"l2": [], "mape": []}
for idata in data.values():
  _, y_true, y_pred = utils.predict(model, idata.to_numpy(), indices, t, t_index=3)
  y_true *= scale
  y_pred *= scale
  error["l2"].append(utils.error_l2(y_true, y_pred, eps=eps))
  error["mape"].append(utils.error_mape(y_true, y_pred, eps=eps))

Plot and save error

In [ ]:
xticks = [1] + list(range(5,46,5))
xticklabels = [str(i) for i in xticks]

In [ ]:
for (k, err) in error.items():
  kerr = np.vstack(err)
  mean, uncert = utils.compute_stats(kerr, axis=0, alpha=alpha)
  kerr = np.vstack([mean, uncert])
  # Plotting error on distribution
  if (model_name != "meno_nocorr"):
    # > Remove temperature and atom
    mean = mean[1:-1]
    uncert = uncert[1:-1]
  utils.plot_err_discrete(
    x=np.arange(len(mean))+1,
    mean=mean,
    uncert=uncert,
    alpha=alpha,
    xlim=[0.5,45.5],
    xticks=xticks,
    xticklabels=xticklabels,
    labels=[r"O$_2$ Groups", r"Error [\%]"],
    scales=["linear", "linear"],
    legend_loc="best",
    figname=paths["saving"]+f"/err_dist_{k}.pdf",
    save=True,
    show=False
  )
  # Saving
  kerr = pd.DataFrame(kerr, columns=names, index=["mean", uncert_name])
  kerr = kerr.sort_values(by=['mean'], axis=1)
  kerr.to_csv(paths["saving"]+f"/{model_name}_{k}.csv", float_format="%.3f")
  utils.write_latex(
    error=kerr,
    filename=paths["saving"]+f"/{model_name}_{k}_latex.txt",
    names=names_latex,
    names_stats=["mean", uncert_name]
  )